In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import matplotlib as mpl

# imports from "common" folder
from common.data_processing import process_data

### 1. Data Processing

In [ ]:
# Data processing from data_processing.py
train, test, y_train, y_test, X_train_scaled, X_test_scaled = process_data()

In [ ]:
traintest = pd.concat([train, test]).sort_index()

In [ ]:
# RF predictions
rf_train = pd.read_csv("../data/RF_Predictions_training_I.csv", index_col=0)
rf_test  = pd.read_csv("../data/RF_Predictions_test_I.csv", index_col=0)

rf_traintest = pd.concat([rf_train, rf_test])

traintest = traintest.copy()

traintest["RF_Predicted"] = rf_traintest.loc[traintest.index, "Predicted"].values

In [ ]:
#  BRT predictions
brt_train = pd.read_csv("../data/BRT_Predictions_training_I.csv", index_col=0)
brt_test  = pd.read_csv("../data/BRT_Predictions_test_I.csv", index_col=0)

brt_traintest = pd.concat([brt_train, brt_test])

traintest["BRT_Predicted"] = brt_traintest.loc[traintest.index, "Predicted"].values


In [ ]:
# MLP predictions
mlp_train = pd.read_csv("../data/MLP_Predictions_training_I.csv", index_col=0)
mlp_test  = pd.read_csv("../data/MLP_Predictions_test_I.csv", index_col=0)

mlp_traintest = pd.concat([mlp_train, mlp_test])

traintest["MLP_Predicted"] = mlp_traintest.loc[traintest.index, "Predicted"].values

In [ ]:
# GAM predictions
gam_train = pd.read_csv("../data/GAM_Predictions_training_I.csv", index_col=0)
gam_test  = pd.read_csv("../data/GAM_Predictions_test_I.csv", index_col=0)

gam_traintest = pd.concat([gam_train, gam_test])

traintest["GAM_Predicted"] = gam_traintest.loc[traintest.index, "Predicted"].values

### 2. Time series of MHCC predictions by the models and the observed MHCC at 10m depth.

In [ ]:
# df_pred = pd.concat([trainval, test], axis=0).reset_index(drop=True)
df_pred = traintest

df_pred = df_pred[[
    "REEF_NAME", "SITE_NO", "year", "mean_hcc",
    "RF_Predicted", "BRT_Predicted", "MLP_Predicted"
    , "GAM_Predicted"
]]

df_pred.rename(columns={"mean_hcc": "Observed"}, inplace=True)


In [ ]:
# Reshape to long format
df_long = df_pred.melt(
    id_vars=["REEF_NAME", "SITE_NO", "year"],
    value_vars=["Observed", "RF_Predicted", "BRT_Predicted", "MLP_Predicted"
                , "GAM_Predicted"
                ],
    var_name="Model",
    value_name="HCC"
)

# Rename the model predictions for visualisation (legend)
df_long["Model"] = df_long["Model"].replace({
    "RF_Predicted": "RF",
    "BRT_Predicted": "BRT",
    "MLP_Predicted": "MLP",
    "GAM_Predicted": "GAM",
    "Observed": "Observed"
})

In [ ]:
reefs = sorted(df_long["REEF_NAME"].unique(),
               key=lambda s: int(re.search(r"\d+", s).group()))

mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    "legend.fontsize": 16,
    "legend.title_fontsize": 14,
    "legend.frameon": False,
})

# Sort reefs for consistent layout
reefs_sorted = sorted(df_long["REEF_NAME"].unique())

palette = {
    "Observed": "black",  
    "GAM":"#C77CFF",      
    "RF": "#00A9FF",       
    "BRT": "#7CAE00",     
    "MLP": "#F8766D",      

}

# FacetGrid
g = sns.FacetGrid(
    df_long,
    col="REEF_NAME",
    col_wrap=7,        
    hue="Model",
    palette=palette, 
    sharey=True,
    height=2.2,
    aspect=1.5,
    col_order=reefs
)

# Add line plots
g.map_dataframe(
    sns.lineplot,
    x="year",
    y="HCC",
    linewidth=1.3,
    errorbar=None,      
    estimator=None,     # do not average across sites
    units="SITE_NO",    # this draws a line per site
    alpha=0.9
)

g.set_titles(col_template="{col_name}")
g.set_axis_labels("Year", "MHCC (%)")

# forecast_year = 30
for ax in g.axes.flatten():
    # ax.axvline(x=forecast_year color='black', linestyle='--', linewidth=1)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.set_xticks(sorted(df_long["year"].unique())[::2])  # every 2nd year
    ax.tick_params(axis='x', rotation=90)

plt.subplots_adjust(top=0.85)

g.add_legend(
    title="",
    loc="upper center",
    bbox_to_anchor=(0.4, 1.03),  
    frameon=False,
    borderaxespad=0
)

g.tight_layout()
plt.show()

In [ ]:
# Additional plot
# Time series plot averaged across sites. 

reefs = sorted(df_long["REEF_NAME"].unique(),
               key=lambda s: int(re.search(r"\d+", s).group()))

# Sort reefs for consistent layout
reefs_sorted = sorted(df_long["REEF_NAME"].unique())


palette = {
    "Observed": "black",   
    "GAM":"#C77CFF",     
    "RF": "#00A9FF",       
    "BRT": "#7CAE00",      
    "MLP": "#F8766D",      
}

# FacetGrid
g = sns.FacetGrid(
    df_long,
    col="REEF_NAME",
    col_wrap=7,       
    hue="Model",
    palette=palette, 
    sharey=True,
    height=2.2,
    aspect=1.5,
    col_order=reefs
)

# Add line plots
g.map_dataframe(
    sns.lineplot,
    x="year",
    y="HCC",
    linewidth= 1.6,
    errorbar=None 
)

g.set_titles(col_template="{col_name}")
g.set_axis_labels("Year", "MHCC (%)")

# forecast_year = 30
for ax in g.axes.flatten():
    # ax.axvline(x=forecast_year color='black', linestyle='--', linewidth=1)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.set_xticks(sorted(df_long["year"].unique())[::2])  # every 2nd year
    ax.tick_params(axis='x', rotation=90)


plt.subplots_adjust(top=0.85)

g.fig.suptitle(
    "Time series of observed vs model predictions",
    fontsize=20,
    y=0.96,  
    x=0.4,  
    ha="center"
)

g.add_legend(
    title="",
    loc="upper center",
    bbox_to_anchor=(0.4, 0.92),   
    ncol=5,
    frameon=False,
    borderaxespad=0
)

plt.show()

### 3. Time series plot of observed and predicted values by the four models for the three representative examples of EDM.

In [ ]:
mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

In [ ]:
# Time series plot of observed and predicted values by the four models for E1 point.

selected_reef = "Reef 35"   
selected_site = "Site 1"  
relevant_year = 20   


# Subset data
df_sel = df_long[
    (df_long["REEF_NAME"] == selected_reef) &
    (df_long["SITE_NO"] == selected_site)
].copy()


palette = {
    "Observed": "black",
    "GAM": "#C77CFF",
    "RF": "#00A9FF",
    "BRT": "#7CAE00",
    "MLP": "#F8766D",
}

fig, ax = plt.subplots(figsize=(8, 3))

sns.lineplot(
    data=df_sel,
    x="year",
    y="HCC",
    hue="Model",
    palette=palette,
    linewidth=1.4,
    ax=ax 
)


# Vertical line 
ax.axvline(x=relevant_year, color='black', linestyle='--', linewidth=1)

ax.grid(True, linestyle="--", alpha=0.4)
ax.set_xlabel("Year")
ax.set_ylabel("MHCC (%)")

ax.set_xticks(sorted(df_sel["year"].unique())[::2])
ax.tick_params(axis='x', rotation=90)

ax.set_title(f"{selected_reef} - {selected_site}")
ax.legend(title="", loc="upper right", bbox_to_anchor=(1.35, 1.0), frameon = False)
plt.show()


In [ ]:
# Time series plot of observed and predicted values by the four models for E2 point.

selected_reef = "Reef 25"   
selected_site = "Site 1"  
relevant_year = 6   

# Subset data
df_sel = df_long[
    (df_long["REEF_NAME"] == selected_reef) &
    (df_long["SITE_NO"] == selected_site)
].copy()


palette = {
    "Observed": "black",
    "GAM": "#C77CFF",
    "RF": "#00A9FF",
    "BRT": "#7CAE00",
    "MLP": "#F8766D",
}

fig, ax = plt.subplots(figsize=(8, 3))

sns.lineplot(
    data=df_sel,
    x="year",
    y="HCC",
    hue="Model",
    palette=palette,
    linewidth=1.4,
    ax=ax 
)

# Vertical line 
ax.axvline(x=relevant_year, color='black', linestyle='--', linewidth=1)

ax.grid(True, linestyle="--", alpha=0.4)
ax.set_xlabel("Year")
ax.set_ylabel("MHCC (%)")

ax.set_xticks(sorted(df_sel["year"].unique())[::2])
ax.tick_params(axis='x', rotation=90)

ax.set_title(f"{selected_reef} - {selected_site}")
ax.legend(title="", loc="upper right", bbox_to_anchor=(1.35, 1.0), frameon = False)
plt.show()


In [ ]:
# Time series plot of observed and predicted values by the four models for E3 point.
selected_reef = "Reef 44"   
selected_site = "Site 2"  
relevant_year = 26   


# Subset data
df_sel = df_long[
    (df_long["REEF_NAME"] == selected_reef) &
    (df_long["SITE_NO"] == selected_site)
].copy()


palette = {
    "Observed": "black",
    "GAM": "#C77CFF",
    "RF": "#00A9FF",
    "BRT": "#7CAE00",
    "MLP": "#F8766D",
}

fig, ax = plt.subplots(figsize=(8, 3))

sns.lineplot(
    data=df_sel,
    x="year",
    y="HCC",
    hue="Model",
    palette=palette,
    linewidth=1.4,
    ax=ax 
)


# Vertical line 
ax.axvline(x=relevant_year, color='black', linestyle='--', linewidth=1)

ax.grid(True, linestyle="--", alpha=0.4)
ax.set_xlabel("Year")
ax.set_ylabel("MHCC (%)")

ax.set_xticks(sorted(df_sel["year"].unique())[::2])
ax.tick_params(axis='x', rotation=90)

ax.set_title(f"{selected_reef} - {selected_site}")

ax.legend(title="", loc="upper right", bbox_to_anchor=(1.35, 1.0), frameon = False)
plt.show()
